# Flash Attention 2 Handwrite Backward

Author： xiaodongguaAIGC

手撕Flash Attention 2，可以直接跳过1学习。

1. 实现attention前向计算
2. 实现attention autograd 梯度计算
3. 实现attention 手写梯度计算
4. 实现Flash Attention 2 除法版本版本
5. 实现Flash Attention 2 非除法计算版本
6. 实现Flash Attention 2 手写backward
7. 分析Flash Attention 2 梯度计算量和通信量

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(42)

In [2]:
dim = 4 # d_model
n = 6 # seq_length
block = 2 # block
nb = n // block # seq per block

In [3]:
X_src = torch.randn(n, dim)
Y_label = torch.randn(n, dim)

# Forward

In [4]:
class attention(nn.Module):
    def __init__(self, dim):
        super(attention, self).__init__()
        self.dim = dim
        self.w = nn.Linear(dim, dim, bias = False)
        self.wq = nn.Linear(dim, dim, bias = False)
        self.wk = nn.Linear(dim, dim, bias = False)
        self.wv = nn.Linear(dim, dim, bias = False)
        self.wo = nn.Linear(dim, dim, bias = False)
        
    def forward(self, X_src):
        X = self.w(X_src)
        Q, K, V = self.wq(X), self.wk(X), self.wv(X)
        S = Q @ K.t() / math.sqrt(dim)
        S_softmax = F.softmax(S, dim = 1)
        O = S_softmax @ V
        Y = self.wo(O)
        return Y, X, Q, K, V, S, S_softmax, O

model = attention(dim)
Y, X, Q, K, V, S, S_softmax, O = model(X_src)
X.retain_grad()
O.retain_grad()
Y.retain_grad()
V.retain_grad()
S_softmax.retain_grad() 
S.retain_grad()
Q.retain_grad()
K.retain_grad()
X.retain_grad()
print(Y)
# 其中 # 可以做成Gradient checkpoint

tensor([[-0.2589,  0.0210, -0.0432,  0.0851],
        [-0.1778,  0.0324, -0.0253,  0.0536],
        [-0.1226,  0.0396, -0.0070,  0.0353],
        [-0.1270,  0.0386, -0.0061,  0.0380],
        [-0.1276,  0.0395, -0.0079,  0.0371],
        [-0.1096,  0.0413, -0.0009,  0.0318]], grad_fn=<MmBackward0>)

# Backward AutoGrad

In [5]:
loss = (0.5 * (Y - Y_label) ** 2).mean() # MSE Loss
loss.backward()
print('grad wo:', model.wo.weight.grad)
print('grad wq:', model.wq.weight.grad)
print('grad wk:', model.wk.weight.grad)
print('grad wv:', model.wv.weight.grad)
print('grad w:', model.w.weight.grad)

grad wo: tensor([[-0.0074, -0.0033,  0.0375,  0.0437],
        [-0.0021, -0.0026,  0.0146,  0.0168],
        [ 0.0013, -0.0059,  0.0084,  0.0104],
        [-0.0004,  0.0041, -0.0079, -0.0098]])

grad wq: tensor([[-0.0010, -0.0092,  0.0019, -0.0047],
        [-0.0010, -0.0095,  0.0020, -0.0049],
        [-0.0007, -0.0051,  0.0011, -0.0029],
        [-0.0007, -0.0080,  0.0018, -0.0040]])

grad wk: tensor([[-4.2545e-04, -5.8015e-03, -1.4515e-04, -2.0063e-03],
        [ 3.7423e-04, -9.9471e-04, -4.5874e-05, -1.0029e-03],
        [-3.7292e-04, -6.8749e-03, -1.1640e-04, -2.3837e-03],
        [-7.1166e-04, -7.7452e-03,  7.3929e-06, -2.1941e-03]])

grad wv: tensor([[-9.4131e-04, -4.2935e-04,  2.3561e-04, -1.7016e-03],
        [-1.4064e-03,  1.2744e-02,  7.3550e-04,  6.8130e-03],
        [-1.4330e-03, -2.2513e-02, -9.7612e-05, -2.0825e-02],
        [-3.7709e-03, -1.4531e-02,  8.2666e-04, -1.9862e-02]])

grad w: tensor([[ 0.0054, -0.0005,  0.0016, -0.0062],
        [ 0.0341,  0.0012,  0.0005, -0.0582],
        [-0.0039,  0.0047,  0.0135,  0.0220],
        [ 0.0217, -0.0060, -0.0023, -0.0355]])

In [6]:
print('grad X:', X.grad)
print('grad O:', O.grad)
print('grad Y:', Y.grad)
print('grad V:', V.grad)
print('grad S_softmax:', S_softmax.grad)
print('grad S:', S.grad)
print('grad Q:', Q.grad)
print('grad K:', K.grad)

grad X: tensor([[ 0.0020,  0.0175, -0.0056,  0.0097],
        [ 0.0015,  0.0123, -0.0053,  0.0095],
        [ 0.0019,  0.0096, -0.0061,  0.0099],
        [-0.0002,  0.0050, -0.0066,  0.0027],
        [ 0.0006,  0.0078, -0.0060,  0.0049],
        [ 0.0002,  0.0057, -0.0058,  0.0043]])

grad O: tensor([[ 0.0094,  0.0357, -0.0291,  0.0120],
        [-0.0049, -0.0093,  0.0035, -0.0401],
        [-0.0102,  0.0378, -0.0667,  0.0205],
        [-0.0100, -0.0314,  0.0076, -0.0121],
        [-0.0135,  0.0064, -0.0059,  0.0105],
        [ 0.0191, -0.0429,  0.0434, -0.0602]])

grad Y: tensor([[-0.0122, -0.0125, -0.0674,  0.0388],
        [-0.0621, -0.0273,  0.0443,  0.0170],
        [-0.0653, -0.0340, -0.0927, -0.0203],
        [-0.0197,  0.0098,  0.0437, -0.0517],
        [ 0.0019, -0.0202, -0.0027, -0.0162],
        [-0.0285,  0.0285,  0.0919,  0.0326]])

grad V: tensor([[-0.0006,  0.0059, -0.0135, -0.0109],
        [-0.0018,  0.0004, -0.0087, -0.0116],
        [-0.0019, -0.0026, -0.0062, -0.0118],
        [-0.0020, -0.0027, -0.0060, -0.0117],
        [-0.0019, -0.0016, -0.0070, -0.0117],
        [-0.0020, -0.0031, -0.0057, -0.0118]])

grad S_softmax: tensor([[ 0.0213,  0.0206, -0.0008, -0.0244,  0.0129, -0.0117],
        [ 0.0389,  0.0133,  0.0038, -0.0008, -0.0072, -0.0055],
        [ 0.0406,  0.0435,  0.0011, -0.0459,  0.0239, -0.0224],
        [-0.0038, -0.0040,  0.0011,  0.0115, -0.0052,  0.0044],
        [-0.0044, -0.0009,  0.0011, -0.0050,  0.0004, -0.0009],
        [ 0.0183, -0.0083,  0.0009,  0.0306, -0.0182,  0.0081]])

grad S: tensor([[ 4.3081e-03,  2.1164e-03, -1.1535e-03, -3.6334e-03,  5.1049e-04,
         -2.1480e-03],
        [ 6.6428e-03,  6.2975e-04, -8.9468e-04, -1.5332e-03, -2.7168e-03,
         -2.1279e-03],
        [ 5.4696e-03,  6.1777e-03, -9.1578e-04, -8.7766e-03,  2.9211e-03,
         -4.8760e-03],
        [-7.6980e-04, -7.4957e-04,  7.5381e-05,  1.7852e-03, -9.6416e-04,
          6.2292e-04],
        [-4.6110e-04,  1.1459e-04,  4.5426e-04, -5.6657e-04,  3.3694e-04,
          1.2188e-04],
        [ 1.9311e-03, -2.1551e-03, -7.5147e-04,  4.4183e-03, -3.9457e-03,
          5.0286e-04]])

grad Q: tensor([[-3.3238e-03, -3.4420e-03, -1.8728e-03, -2.9382e-03],
        [-3.7576e-03, -4.0795e-03, -2.6218e-03, -3.0564e-03],
        [-5.7109e-03, -5.7855e-03, -2.8001e-03, -5.3474e-03],
        [ 8.7532e-04,  8.5315e-04,  3.5671e-04,  8.0495e-04],
        [ 1.5613e-04,  1.9462e-04,  1.7920e-04,  1.2406e-04],
        [ 2.2472e-04, -2.0580e-05, -5.5875e-04,  4.6215e-04]])

grad K: tensor([[-2.0525e-03, -6.7350e-04, -2.5053e-03, -2.5500e-03],
        [-8.8267e-04, -2.4275e-05, -8.8704e-04, -1.1448e-03],
        [ 6.3575e-04,  1.6152e-04,  5.2043e-04,  2.6962e-04],
        [ 1.4340e-03,  1.4235e-05,  1.8325e-03,  2.2755e-03],
        [-8.6057e-05,  3.4232e-04, -2.3270e-05, -1.2448e-04],
        [ 9.5143e-04,  1.7970e-04,  1.0627e-03,  1.2742e-03]])

# Backward Hand-write

In [7]:
N = n * dim 

In [8]:
dY =  Y - Y_label
dY = 1 / N * dY
print(dY)

tensor([[-0.0122, -0.0125, -0.0674,  0.0388],
        [-0.0621, -0.0273,  0.0443,  0.0170],
        [-0.0653, -0.0340, -0.0927, -0.0203],
        [-0.0197,  0.0098,  0.0437, -0.0517],
        [ 0.0019, -0.0202, -0.0027, -0.0162],
        [-0.0285,  0.0285,  0.0919,  0.0326]], grad_fn=<MulBackward0>)

In [9]:
dwo = O.t() @ dY 
print(dwo)

tensor([[-0.0074, -0.0021,  0.0013, -0.0004],
        [-0.0033, -0.0026, -0.0059,  0.0041],
        [ 0.0375,  0.0146,  0.0084, -0.0079],
        [ 0.0437,  0.0168,  0.0104, -0.0098]], grad_fn=<MmBackward0>)

In [10]:
# Y = O @ Wo     n x d = n x d @ d x d 
dO = dY @ model.wo.weight
print(dO)

tensor([[ 0.0094,  0.0357, -0.0291,  0.0120],
        [-0.0049, -0.0093,  0.0035, -0.0401],
        [-0.0102,  0.0378, -0.0667,  0.0205],
        [-0.0100, -0.0314,  0.0076, -0.0121],
        [-0.0135,  0.0064, -0.0059,  0.0105],
        [ 0.0191, -0.0429,  0.0434, -0.0602]], grad_fn=<MmBackward0>)

In [11]:
# O = S_softmax @ V    n x d = n x n @ n x d
dV = S_softmax.t() @ dO
print(dV)

tensor([[-0.0006,  0.0059, -0.0135, -0.0109],
        [-0.0018,  0.0004, -0.0087, -0.0116],
        [-0.0019, -0.0026, -0.0062, -0.0118],
        [-0.0020, -0.0027, -0.0060, -0.0117],
        [-0.0019, -0.0016, -0.0070, -0.0117],
        [-0.0020, -0.0031, -0.0057, -0.0118]], grad_fn=<MmBackward0>)

In [12]:
# V = Wv @ X    # n x d = n x d @ d x d
dwv = dV.t() @ X
print(dwv)

tensor([[-9.4131e-04, -4.2935e-04,  2.3561e-04, -1.7016e-03],
        [-1.4064e-03,  1.2744e-02,  7.3550e-04,  6.8130e-03],
        [-1.4330e-03, -2.2513e-02, -9.7612e-05, -2.0825e-02],
        [-3.7709e-03, -1.4531e-02,  8.2666e-04, -1.9862e-02]],
       grad_fn=<MmBackward0>)

In [13]:
# O = S_softmax @ V
dS_softmax = dO @ V.t()
print(dS_softmax)

tensor([[ 0.0213,  0.0206, -0.0008, -0.0244,  0.0129, -0.0117],
        [ 0.0389,  0.0133,  0.0038, -0.0008, -0.0072, -0.0055],
        [ 0.0406,  0.0435,  0.0011, -0.0459,  0.0239, -0.0224],
        [-0.0038, -0.0040,  0.0011,  0.0115, -0.0052,  0.0044],
        [-0.0044, -0.0009,  0.0011, -0.0050,  0.0004, -0.0009],
        [ 0.0183, -0.0083,  0.0009,  0.0306, -0.0182,  0.0081]],
       grad_fn=<MmBackward0>)

In [14]:
# S_softmax = softmax(S)
# gradient = diag(S_softmax) - S_softmax X S_softmax

dS = torch.zeros_like(S_softmax)
for i in range(n):
    I = torch.diag(S_softmax[i,:]) - torch.outer(S_softmax[i,:], S_softmax[i,:])
    dS[i,:] = dS_softmax[i,:] @ I
print(dS)

tensor([[ 4.3081e-03,  2.1164e-03, -1.1535e-03, -3.6334e-03,  5.1049e-04,
         -2.1480e-03],
        [ 6.6428e-03,  6.2975e-04, -8.9468e-04, -1.5332e-03, -2.7168e-03,
         -2.1279e-03],
        [ 5.4696e-03,  6.1777e-03, -9.1578e-04, -8.7766e-03,  2.9211e-03,
         -4.8760e-03],
        [-7.6980e-04, -7.4957e-04,  7.5381e-05,  1.7852e-03, -9.6416e-04,
          6.2292e-04],
        [-4.6110e-04,  1.1459e-04,  4.5426e-04, -5.6657e-04,  3.3694e-04,
          1.2188e-04],
        [ 1.9311e-03, -2.1551e-03, -7.5147e-04,  4.4183e-03, -3.9457e-03,
          5.0286e-04]], grad_fn=<CopySlices>)

In [15]:
# S = Q @ K.t() / sqrt(dim)
dQ = dS @ K  / math.sqrt(dim)
print(dQ)

tensor([[-3.3238e-03, -3.4420e-03, -1.8728e-03, -2.9382e-03],
        [-3.7576e-03, -4.0795e-03, -2.6218e-03, -3.0563e-03],
        [-5.7109e-03, -5.7855e-03, -2.8001e-03, -5.3474e-03],
        [ 8.7532e-04,  8.5315e-04,  3.5671e-04,  8.0495e-04],
        [ 1.5613e-04,  1.9462e-04,  1.7920e-04,  1.2406e-04],
        [ 2.2472e-04, -2.0580e-05, -5.5875e-04,  4.6215e-04]],
       grad_fn=<DivBackward0>)

In [16]:
# Q = wq @ X
dwq = dQ.t() @ X
print(dwq)

tensor([[-0.0010, -0.0092,  0.0019, -0.0047],
        [-0.0010, -0.0095,  0.0020, -0.0049],
        [-0.0007, -0.0051,  0.0011, -0.0029],
        [-0.0007, -0.0080,  0.0018, -0.0040]], grad_fn=<MmBackward0>)

In [17]:
# S = Q @ d.t() / sqrt(dim)
dK = dS.t() @ Q / math.sqrt(dim)
print(dK)

tensor([[-2.0525e-03, -6.7350e-04, -2.5053e-03, -2.5500e-03],
        [-8.8267e-04, -2.4275e-05, -8.8704e-04, -1.1448e-03],
        [ 6.3575e-04,  1.6152e-04,  5.2043e-04,  2.6962e-04],
        [ 1.4340e-03,  1.4235e-05,  1.8325e-03,  2.2755e-03],
        [-8.6057e-05,  3.4232e-04, -2.3270e-05, -1.2448e-04],
        [ 9.5143e-04,  1.7970e-04,  1.0627e-03,  1.2742e-03]],
       grad_fn=<DivBackward0>)

In [18]:
# K = wk @ X
dwk = dK.t() @ X
print(dwk)

tensor([[-4.2545e-04, -5.8015e-03, -1.4515e-04, -2.0063e-03],
        [ 3.7423e-04, -9.9471e-04, -4.5874e-05, -1.0029e-03],
        [-3.7292e-04, -6.8749e-03, -1.1640e-04, -2.3837e-03],
        [-7.1166e-04, -7.7452e-03,  7.3930e-06, -2.1941e-03]],
       grad_fn=<MmBackward0>)

In [19]:
# Q, K, V = self.wq(X), self.wk(X), self.wv(X)
dXQ = dQ @ model.wq.weight
dXK = dK @ model.wk.weight
dXV = dV @ model.wv.weight
dX = dXQ + dXK + dXV
print(dX)

tensor([[ 0.0020,  0.0175, -0.0056,  0.0097],
        [ 0.0015,  0.0123, -0.0053,  0.0095],
        [ 0.0019,  0.0096, -0.0061,  0.0099],
        [-0.0002,  0.0050, -0.0066,  0.0027],
        [ 0.0006,  0.0078, -0.0060,  0.0049],
        [ 0.0002,  0.0057, -0.0058,  0.0043]], grad_fn=<AddBackward0>)

In [20]:
# X = W X_Src
dw = dX.t() @ X_src
print(dw)

tensor([[ 0.0054, -0.0005,  0.0016, -0.0062],
        [ 0.0341,  0.0012,  0.0005, -0.0582],
        [-0.0039,  0.0047,  0.0135,  0.0220],
        [ 0.0217, -0.0060, -0.0023, -0.0355]], grad_fn=<MmBackward0>)

小结：

1. 为了帮助我们求attention的梯度，那么我们要把前向计算过程的中间状态都保留下来，这些就是Gradient Checkpoint
2. 而Gradient Checkpoint带来显著的显存消耗，如何进一步优化？
3. 计算Attention得梯度主要区分两种：1. 乘法， 2. Softmax？
4. 

# Flash Attention Forward

## inner scaled

先实现带scale版本

In [21]:
def flash_attention(Q, K, V):
    O = torch.zeros_like(Q)
    
    for tq in range(block):     # q loop
        q = Q[tq*nb : (tq+1)*nb, :]
        o_old = torch.zeros_like(q)
    
        # global
        l_old = m_old = torch.zeros(nb, 1)
    
        # scale cache
        l_cache = torch.ones(nb, 1)
        
        for tk in range(block): # kv loop
            k = K[tk*nb : (tk+1)*nb , :]
            v = V[tk*nb : (tk+1)*nb , :]
    
            # score
            s = q @ k.t() / math.sqrt(dim)
    
            # local 
            m = torch.max(s, dim = 1, keepdim = True).values
            m_new = torch.maximum(m, m_old)
            l = torch.sum(torch.exp(s - m_new) , dim = 1, keepdim = True)

            # update global 
            l_new = l_old * torch.exp(m_old - m_new) + l
            l_cache = l_cache * l_new
    
            # update o
            o =  l_old * o_old * torch.exp(m_old - m_new) + torch.exp(s - m_new) @ v
            o = o / l_new
    
            # replace
            o_old = o
            l_old = l_new
            m_old = m_new

        O[tq*nb: (tq+1)*nb, :] = o
        # break

    return O

flash_attention(Q, K, V) 

tensor([[ 0.0292,  0.1025, -0.3566, -0.4337],
        [ 0.0392,  0.0383, -0.2478, -0.2834],
        [ 0.0416, -0.0016, -0.1643, -0.1894],
        [ 0.0392,  0.0033, -0.1672, -0.2001],
        [ 0.0415,  0.0013, -0.1710, -0.1982],
        [ 0.0410, -0.0104, -0.1419, -0.1693]], grad_fn=<CopySlices>)

## out scaled 

外循环scale版本

In [22]:
def flash_attention(Q, K, V):
    O = torch.zeros_like(Q)
    L = torch.zeros(n, 1)
    
    for tq in range(block):     # q loop
        q = Q[tq*nb : (tq+1)*nb, :]
        o_old = torch.zeros_like(q)
    
        # global
        l_old = m_old = torch.zeros(nb, 1)
    
        # scale cache
        l_cache = torch.ones(nb, 1)
        
        for tk in range(block): # kv loop
            k = K[tk*nb : (tk+1)*nb , :]
            v = V[tk*nb : (tk+1)*nb , :]
    
            # score
            s = q @ k.t() / math.sqrt(dim)
    
            # local 
            m = torch.max(s, dim = 1, keepdim = True).values
            m_new = torch.maximum(m, m_old)
            l = torch.sum(torch.exp(s - m_new) , dim = 1, keepdim = True)
            print(l)
            print(m)

            # update global 
            l_new = l_old * torch.exp(m_old - m_new) + l
            l_cache = l_cache * l_new
    
            # update o
            o =  o_old * torch.exp(m_old - m_new) + torch.exp(s - m_new) @ v
            # o = o / l_new
    
            # replace
            o_old = o
            l_old = l_new
            m_old = m_new

        o = o_old / l_old
        O[tq*nb: (tq+1)*nb, :] = o
        L[tq*nb: (tq+1)*nb, :] = m_old + (l_old).log() # 要存储这个L，用于后续backward

    return O, L

O, L = flash_attention(Q, K, V) 

tensor([[1.8469],
        [2.4437],
        [2.9562]], grad_fn=<SumBackward1>)

tensor([[1.0351e+00],
        [4.3011e-01],
        [8.6359e-04]], grad_fn=<MaxBackward0>)

tensor([[0.9762],
        [1.9127],
        [2.9841]], grad_fn=<SumBackward1>)

tensor([[0.0673],
        [0.0608],
        [0.0099]], grad_fn=<MaxBackward0>)

tensor([[2.9176],
        [2.9344],
        [2.7628]], grad_fn=<SumBackward1>)

tensor([[0.0137],
        [0.0424],
        [0.0109]], grad_fn=<MaxBackward0>)

tensor([[2.9004],
        [2.9121],
        [2.9357]], grad_fn=<SumBackward1>)

tensor([[-0.0054],
        [ 0.0237],
        [ 0.0285]], grad_fn=<MaxBackward0>)

In [23]:
Y = model.wo(O)
print(Y)

tensor([[-0.2589,  0.0210, -0.0432,  0.0851],
        [-0.1778,  0.0324, -0.0253,  0.0536],
        [-0.1226,  0.0396, -0.0070,  0.0353],
        [-0.1270,  0.0386, -0.0061,  0.0380],
        [-0.1276,  0.0395, -0.0079,  0.0371],
        [-0.1096,  0.0413, -0.0009,  0.0318]], grad_fn=<MmBackward0>)

# Backward

Flash attention2 的 前向计算是迭代式子算M、L、O的

1. 那么在Backward时，是否需要迭代式的求 M L O 呢？
2. 我们如果认为forward时的softmax是一种非线性计算，那么反向时算是线性计算还是非线性计算，是否非线性计算，就不用迭代式的M、L、O了呢？
3. 为了高效计算Backward，Forward时要保留哪些数据，便于计算梯度？
4. backward计算时内外循环是怎么区安排先Q再K，还是先K再Q， 从数据的流向进一步分析SRAM和HBM之间的交换效率
5. 在长文本中，我们的检索有超长的KV，那么如何来做设计分布式序列并行训练算法呢？
6. 我们在单个GPU上，能否并行的计算 (Q K1,k2,k3,k4), (Q K5,k6,k7,k8), (Q K9,k10,k11,k12)？
7. FlashAttention 和 Page Attention 之间有什么联系？

In [24]:
O,L = flash_attention(Q, K, V)
print(O.shape)
print(L.shape)

tensor([[1.8469],
        [2.4437],
        [2.9562]], grad_fn=<SumBackward1>)

tensor([[1.0351e+00],
        [4.3011e-01],
        [8.6359e-04]], grad_fn=<MaxBackward0>)

tensor([[0.9762],
        [1.9127],
        [2.9841]], grad_fn=<SumBackward1>)

tensor([[0.0673],
        [0.0608],
        [0.0099]], grad_fn=<MaxBackward0>)

tensor([[2.9176],
        [2.9344],
        [2.7628]], grad_fn=<SumBackward1>)

tensor([[0.0137],
        [0.0424],
        [0.0109]], grad_fn=<MaxBackward0>)

tensor([[2.9004],
        [2.9121],
        [2.9357]], grad_fn=<SumBackward1>)

tensor([[-0.0054],
        [ 0.0237],
        [ 0.0285]], grad_fn=<MaxBackward0>)

torch.Size([6, 4])

torch.Size([6, 1])

# Prepare data

In [25]:
N = n * dim
Y = model.wo(O)
print(Y)
dY =  Y - Y_label
dY = 1 / N * dY
print(dY)

tensor([[-0.2589,  0.0210, -0.0432,  0.0851],
        [-0.1778,  0.0324, -0.0253,  0.0536],
        [-0.1226,  0.0396, -0.0070,  0.0353],
        [-0.1270,  0.0386, -0.0061,  0.0380],
        [-0.1276,  0.0395, -0.0079,  0.0371],
        [-0.1096,  0.0413, -0.0009,  0.0318]], grad_fn=<MmBackward0>)

tensor([[-0.0122, -0.0125, -0.0674,  0.0388],
        [-0.0621, -0.0273,  0.0443,  0.0170],
        [-0.0653, -0.0340, -0.0927, -0.0203],
        [-0.0197,  0.0098,  0.0437, -0.0517],
        [ 0.0019, -0.0202, -0.0027, -0.0162],
        [-0.0285,  0.0285,  0.0919,  0.0326]], grad_fn=<MulBackward0>)

# Flash Attention 2 backward handwrite

In [26]:
dwo = O.t() @ dY
print(dwo)

tensor([[-0.0074, -0.0021,  0.0013, -0.0004],
        [-0.0033, -0.0026, -0.0059,  0.0041],
        [ 0.0375,  0.0146,  0.0084, -0.0079],
        [ 0.0437,  0.0168,  0.0104, -0.0098]], grad_fn=<MmBackward0>)

In [27]:
dO = dY @ model.wo.weight
print(dO)

tensor([[ 0.0094,  0.0357, -0.0291,  0.0120],
        [-0.0049, -0.0093,  0.0035, -0.0401],
        [-0.0102,  0.0378, -0.0667,  0.0205],
        [-0.0100, -0.0314,  0.0076, -0.0121],
        [-0.0135,  0.0064, -0.0059,  0.0105],
        [ 0.0191, -0.0429,  0.0434, -0.0602]], grad_fn=<MmBackward0>)

## analysis

当我们有Q, K, V, O L dO 后就可以计算Flash Attention 2 的 backward了

1. 整体计算结构为先KV 再Q
2. 注意到S和P是需要recompute， 但是P的计算需要 sum(S)和max(S), 但是这里的分块计算时，可以拿Forward的全局信息计算。
3. 在Forward 保存的是L = M+log(l)， 则 backward时的softmax计算为：
```
       e(sij-Li) = e(sij) / e(L) = e(sij) / e(M + log(l))
                                 = e(sij) / (e(M) * log(l))
                                 = (e(sij) / e(M)) * loge(l)
                                 = e(sij-M) / l
```

4. backward时不存在迭代计算
5. 以下公式里的 V <- V + xxxx, 这里不是迭代，而是求和。
6. 在前向时Q驻留在SRAM， 反向时kv, dKV驻留在SRAM
7. 在Flash attention 中，可以构造以切分块数为变量，分析Flash Attention加速比。 

![backward](./flash_attention_v2_backward.jpeg)

## implemention

In [47]:
print(block)
def flash_attention_backward_D(Q, K, V, O, dO, L):
    dQ = torch.zeros_like(Q)
    dK = torch.zeros_like(Q)
    dV = torch.zeros_like(Q)
    dS = torch.zeros(n, n)
    dP = torch.zeros(n, n)
    D = torch.sum(O * dO, dim = 1, keepdim=True)

    for tk in range(block): # kv loop
        k = K[tk*nb : (tk+1)*nb , :]
        v = V[tk*nb : (tk+1)*nb , :]

            
        for tq in range(block):     # q loop
            q = Q[tq*nb : (tq+1)*nb, :]
            l = L[tq*nb : (tq+1)*nb, :]            
            do = dO[tq*nb : (tq+1)*nb , :]
            d = D[tq*nb : (tq+1)*nb , :]
    
            # # score
            s = q @ k.t() / math.sqrt(dim)
            p = torch.exp(s - l) # forward softmax
            dv = p.t() @ do
            dV[tk*nb : (tk+1)*nb, :] = dV[tk*nb : (tk+1)*nb, :] + dv

            # dP
            dp = do @ v.t()
            dP[tq*nb : (tq+1)*nb, tk*nb : (tk+1)*nb] = dp
            
            ds = p * (dp - d) 
            dS[tq*nb : (tq+1)*nb, tk*nb : (tk+1)*nb] = ds
                
            # # dQ, dK
            dq = ds @ k / math.sqrt(dim)
            dk = ds.t() @ q / math.sqrt(dim)
            dK[tk*nb : (tk+1)*nb, :] = dK[tk*nb : (tk+1)*nb, :] + dk
            dQ[tq*nb : (tq+1)*nb, :] = dQ[tq*nb : (tq+1)*nb, :] + dq

    return  dQ, dK, dV, dS, dP

dQ, dK, dV, dS, dP = flash_attention_backward_D(Q, K, V, O, dO, L)

2

tensor([[-3.3238e-03, -3.4420e-03, -1.8728e-03, -2.9382e-03],
        [-3.7576e-03, -4.0795e-03, -2.6218e-03, -3.0564e-03],
        [-5.7109e-03, -5.7855e-03, -2.8001e-03, -5.3474e-03],
        [ 8.7532e-04,  8.5315e-04,  3.5671e-04,  8.0495e-04],
        [ 1.5613e-04,  1.9462e-04,  1.7920e-04,  1.2406e-04],
        [ 2.2472e-04, -2.0580e-05, -5.5875e-04,  4.6215e-04]],
       grad_fn=<CopySlices>)

tensor([[-2.0525e-03, -6.7350e-04, -2.5053e-03, -2.5500e-03],
        [-8.8267e-04, -2.4275e-05, -8.8704e-04, -1.1448e-03],
        [ 6.3575e-04,  1.6152e-04,  5.2043e-04,  2.6962e-04],
        [ 1.4340e-03,  1.4235e-05,  1.8325e-03,  2.2755e-03],
        [-8.6057e-05,  3.4232e-04, -2.3270e-05, -1.2448e-04],
        [ 9.5143e-04,  1.7970e-04,  1.0627e-03,  1.2742e-03]],
       grad_fn=<CopySlices>)

tensor([[-0.0006,  0.0059, -0.0135, -0.0109],
        [-0.0018,  0.0004, -0.0087, -0.0116],
        [-0.0019, -0.0026, -0.0062, -0.0118],
        [-0.0020, -0.0027, -0.0060, -0.0117],
        [-0.0019, -0.0016, -0.0070, -0.0117],
        [-0.0020, -0.0031, -0.0057, -0.0118]], grad_fn=<CopySlices>)

tensor([[ 4.3081e-03,  2.1164e-03, -1.1535e-03, -3.6334e-03,  5.1049e-04,
         -2.1480e-03],
        [ 6.6428e-03,  6.2975e-04, -8.9468e-04, -1.5332e-03, -2.7168e-03,
         -2.1279e-03],
        [ 5.4696e-03,  6.1777e-03, -9.1579e-04, -8.7766e-03,  2.9211e-03,
         -4.8760e-03],
        [-7.6980e-04, -7.4957e-04,  7.5381e-05,  1.7852e-03, -9.6416e-04,
          6.2292e-04],
        [-4.6110e-04,  1.1459e-04,  4.5426e-04, -5.6657e-04,  3.3694e-04,
          1.2188e-04],
        [ 1.9311e-03, -2.1551e-03, -7.5147e-04,  4.4183e-03, -3.9457e-03,
          5.0286e-04]], grad_fn=<CopySlices>)

tensor([[ 0.0213,  0.0206, -0.0008, -0.0244,  0.0129, -0.0117],
        [ 0.0389,  0.0133,  0.0038, -0.0008, -0.0072, -0.0055],
        [ 0.0406,  0.0435,  0.0011, -0.0459,  0.0239, -0.0224],
        [-0.0038, -0.0040,  0.0011,  0.0115, -0.0052,  0.0044],
        [-0.0044, -0.0009,  0.0011, -0.0050,  0.0004, -0.0009],
        [ 0.0183, -0.0083,  0.0009,  0.0306, -0.0182,  0.0081]],
       grad_fn=<CopySlices>)